# 04. Dataset и DataLoader — ч.1 (практика)

Видео: [4. Подготовка данных ч.1. Dataset и DataLoader](https://www.youtube.com/watch?v=pm_hLYigX0c&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=5)

**Как смотреть.** Видео короткое (~9 мин) и почти без кода — лучше **целиком**, потом этот ноутбук. Если удобнее кусками: пауза на таймкодах в заголовках блоков.

**Как решать.** В ячейке задания замени `...` на код → запусти проверку под ним. Решения — в `04_dataset_dataloader_1_решения.ipynb`, только после своей попытки.

В видео 4 только теория, поэтому и задачи здесь на теорию: классы, баланс, one-hot, формы батча. Свой `Dataset` и `DataLoader` — в ноутбуке `05` после видео 5.

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)
print(torch.__version__)


def need(cond, msg):
    assert cond, msg


def need_tensor(x, name):
    assert isinstance(x, torch.Tensor), f"{name} — пока не тензор (замени ... на код)"

2.14.1+cpu


## 1. Как лежат размеченные данные
*видео ~0:30–2:05*

**Теория.** Сырые данные на диске ≠ то, что ест сеть. Сначала нужна структура «пример → ответ».

| задача | типичное хранение |
|:---|:---|
| табличная регрессия / классификация | один CSV / parquet: признаки + колонка ответа |
| классификация картинок | папка на класс: `cats/…`, `dogs/…` (имя папки = класс) |
| сегментация | `images/` + `masks/` (или json с путями) |

Для классификации картинок папки по классам удобнее одного json: и человеку видно, и `ImageFolder` (видео 6) подхватит сам.

**1.1** Дан список путей к файлам. Построй словарь `class_to_idx`: имя класса → индекс (порядок — **отсортированные** уникальные имена классов). Индекс потом станет позицией единицы в one-hot.

In [2]:
paths = [
    "data/train/cat/a.png",
    "data/train/dog/b.png",
    "data/train/cat/c.png",
    "data/train/bird/d.png",
]

# подсказка: class_name = path.split("/")[-2]

In [5]:
class_names = ...
class_to_idx = ...
print(class_to_idx)

Ellipsis


In [4]:
need(isinstance(class_to_idx, dict), "class_to_idx должен быть словарём")
need(class_to_idx == {"bird": 0, "cat": 1, "dog": 2}, f"ожидали {{'bird': 0, 'cat': 1, 'dog': 2}}, получили {class_to_idx}")
print("1.1 — верно")

AssertionError: class_to_idx должен быть словарём

## 2. Сбалансированность и репрезентативность
*видео ~2:36–4:11*

**Теория.**
- **Сбалансированный** набор: в каждом классе примерно одинаковое число примеров. Сильный перекос → сеть «любит» частые классы.
- **Репрезентативный** класс: внутри класса примеры **разные** (разные семёрки, разные почерки). Плохое решение дисбаланса — копировать одну и ту же картинку много раз: на train всё ок, на новых семёрках сеть ломается.

MNIST в целом ок по обоим пунктам (лёгкий перекос по некоторым цифрам — норма).

**2.1** Посчитай `n_per_class` (тензор длины 3) — сколько раз встречается каждый класс `0, 1, 2` в `y_toy`. Затем `imbalance_ratio` = max / min (обычное float-деление).

In [ ]:
y_toy = torch.tensor([0, 0, 1, 2, 0, 1, 0, 0, 2, 0])  # специально кривой баланс

In [ ]:
n_per_class = ...
imbalance_ratio = ...
print(n_per_class, imbalance_ratio)

In [ ]:
need_tensor(n_per_class, "n_per_class")
need(tuple(n_per_class.tolist()) == (6, 2, 2), f"счётчики {tuple(n_per_class.tolist())}, ждали (6, 2, 2)")
need(abs(imbalance_ratio - 3.0) < 1e-6, f"ratio={imbalance_ratio}, ждали 3.0")
print("2.1 — верно")

## 3. One-hot: ответ для классификации
*видео ~5:43–6:43*

**Теория.** Класс «7» сеть не выдаёт одним числом «семёрка». На выходе — вектор длины `n_classes`: везде 0, на позиции класса — 1. Это **one-hot**.

```text
класс 7 при 10 цифрах → [0,0,0,0,0,0,0,1,0,0]
класс 0               → [1,0,0,0,0,0,0,0,0,0]
```

Индекс класса (целое `0…9`) удобен внутри `Dataset`. One-hot часто делают позже (loss вроде `CrossEntropyLoss` сам ест индексы — это будет в следующих видео). Сейчас упражняем представление из ролика.

**3.1** Напиши `to_one_hot(idx, n_classes)` → тензор формы `(n_classes,)` типа `float32`.

In [ ]:
def to_one_hot(idx, n_classes):
    ...


v7 = to_one_hot(7, 10)
v0 = to_one_hot(0, 10)
print(v7)
print(v0)

In [ ]:
need_tensor(v7, "v7")
need(v7.shape == (10,) and v7.dtype == torch.float32, f"форма/тип v7: {v7.shape}, {v7.dtype}")
need(v7.tolist() == [0, 0, 0, 0, 0, 0, 0, 1, 0, 0], f"v7={v7.tolist()}")
need(v0.tolist() == [1, 0, 0, 0, 0, 0, 0, 0, 0, 0], f"v0={v0.tolist()}")
print("3.1 — верно")

**3.2** Дан батч индексов `y_batch` формы `(B,)`. Сделай матрицу `Y` формы `(B, n_classes)` one-hot (`float32`).

Можно циклом через `to_one_hot` + `torch.stack`, либо `torch.nn.functional.one_hot(...).float()`.

In [ ]:
y_batch = torch.tensor([7, 0, 3, 9])
n_classes = 10

In [ ]:
Y = ...
print(Y.shape)
print(Y)

In [ ]:
need_tensor(Y, "Y")
need(Y.shape == (4, 10), f"форма {tuple(Y.shape)}, нужна (4, 10)")
need(Y.dtype == torch.float32, f"dtype {Y.dtype}")
need(torch.equal(Y.argmax(dim=1), y_batch), "argmax по строкам должен вернуть исходные индексы")
need(torch.allclose(Y.sum(dim=1), torch.ones(4)), "в каждой строке ровно одна единица")
print("3.2 — верно")

## 4. Формы: что входит в сеть и что выходит
*видео ~7:15–8:00*

**Теория (классификация MNIST-подобных цифр).**

| режим входа | форма батча `X` | форма ответа `Y` (one-hot) |
|:---|:---|:---|
| картинку **вытянули** в вектор 28×28=784 | `(B, 784)` | `(B, 10)` |
| оставили как картинку | `(B, C, H, W)` = `(B, 1, 28, 28)` | `(B, 10)` |

`B` — `batch_size`, `C` — каналы (1 = серое, 3 = RGB).

Для **регрессии** (квадрат на картинке, предсказать центр) ответ — не one-hot, а координаты, например `(B, 2)`.

**4.1** Заполни формы кортежами чисел (не строками).

In [ ]:
B, C, H, W, n_classes = 32, 1, 28, 28, 10

shape_flat_X = ...          # вытянутые картинки
shape_image_X = ...         # как «картинка» с каналом
shape_onehot_Y = ...        # one-hot ответы
shape_reg_Y = ...           # регрессия: (x, y) центра, тот же B

print(shape_flat_X, shape_image_X, shape_onehot_Y, shape_reg_Y)

In [ ]:
need(shape_flat_X == (32, 784), f"flat X: {shape_flat_X}")
need(shape_image_X == (32, 1, 28, 28), f"image X: {shape_image_X}")
need(shape_onehot_Y == (32, 10), f"onehot Y: {shape_onehot_Y}")
need(shape_reg_Y == (32, 2), f"reg Y: {shape_reg_Y}")
print("4.1 — верно")

## 5. Что унести из видео 4

1. Размеченные данные на диске бывают по-разному; для классификации картинок удобны **папки-классы**.
2. Смотри на **баланс** и **разнообразие** внутри класса.
3. Классификация: ответ можно представить как **one-hot** длины `n_classes`.
4. Формы батча: `(B, 784)` или `(B, C, H, W)` на входе, `(B, n_classes)` на выходе; для регрессии координат — `(B, 2)`.

**Дальше:** ноутбук `05` — свой `Dataset`, `DataLoader`, `random_split`.